# Credora portfolio exploration

Explore source integrity, observed outcomes and lending segments. Personal contacts and names are never displayed. All default rates use observed disbursed outcomes; non-disbursed NULLs are not negative labels.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
OUT = ROOT / 'artifacts'
assert json.loads((OUT / 'run_manifest.json').read_text())['version'] == 2, 'Run python -m src.pipeline first'

from src.data_load import read_raw
frames = read_raw()
display(pd.DataFrame({name:{'rows':len(d),'columns':len(d.columns),'missing_cells':int(d.isna().sum().sum())} for name,d in frames.items()}).T)

## Observed outcome population
Approval denominator excludes pending decisions. Default denominator includes only disbursed records with observed outcomes. Timing issues and duplicate optional contacts are recorded in the quality report.

In [ ]:
loans=frames['loan_applications']
display(loans.groupby(['approval_status','disbursed_flag']).agg(applications=('application_id','size'),observed_outcomes=('loan_default','count'),observed_default_rate=('loan_default','mean')))
print(json.loads((OUT/'data_quality.json').read_text())['timing'])
print(json.loads((OUT/'portfolio_summary.json').read_text()))

## Exploratory distributions
Bureau associations use the single June 30 snapshot and are retrospective. They are not historical inputs to the strict applicant model.

In [ ]:
for file in ['credit_score.png','requested_amount.png','debt_to_income.png','default_by_loan.png','correlations.png','monthly_decisions.png']:
    display(Image(filename=str(OUT/'figures'/file)))

## SQL segment evidence
Inspect group counts alongside rates. A higher observed segment rate does not imply a causal effect of loan type or establish how rejected applicants would repay.

In [ ]:
for i in [1,2,3,5,7,10]:
    print(f'Query {i:02d}')
    display(pd.read_csv(OUT/'sql'/f'q{i:02d}.csv').head(12))

## Data limitations
Bureau history cannot be backdated. Customer profiles lack effective dates, 124 applications imply an age below 18 from the supplied birth dates, and repayment outcomes lack event dates and maturity horizons. These are source limitations, not facts imputation can recover.